**EEG Foundations, Signal Processing and Motor Imagery Analysis | Presented by: Reza Saadatyar (2026)**

**01 | EEG Foundations and Motor Imagery**

**Objective:** Introduce the physiological and computational foundations of electroencephalography (EEG) and motor imagery (MI), then inspect the structure and quality of an existing EEG recording before any preprocessing or decoding.

**Dataset:** [BCI Competition IV, Dataset 1](https://www.bbci.de/competition/iv/desc_1.html), using an existing 100 Hz calibration `.mat` file in this repository. The actual classes, channel labels, sampling rate and event annotations are read from the selected file; they are not assumed to be identical across recordings.

**Notebook Scope:** Explain key concepts, locate and inspect the MATLAB source, interpret recording metadata, visualise example EEG channels and cue timing, and perform a stage-specific Scientific Quality Gate. No temporal/spatial filter, feature extraction, classification model or performance benchmark is run here.

**Development status:** This is a newly authored Notebook. Its scientific outputs must be generated by executing the code with the actual dataset. A stage-specific SVG will be added after execution and scientific review.

**Before You Begin | EEG and Motor Imagery Concepts**

EEG measures changes in electrical potential at electrodes on the scalp. Each channel represents a voltage difference relative to a reference, not a direct measurement of individual neuronal action potentials. EEG offers high temporal resolution, but scalp measurements mix activity from multiple neural and non-neural sources.

**Motor imagery (MI)** means mentally rehearsing an action without intentionally performing it. MI experiments commonly use visual or auditory cues to indicate which movement to imagine. EEG-based brain–computer interfaces (BCIs) try to detect task-related activity patterns; imagined movement is **not** the same as measured physical movement.

| Term | Meaning and relevance |
|---|---|
| Sampling frequency (`fs`) | Number of samples recorded each second (Hz); determines the sample-to-time conversion. |
| Channel | One recorded EEG voltage time series, with a sensor label and an acquisition reference. |
| Event/cue | A timestamped instruction or annotation that may mark the start of an MI task. |
| Trial/epoch | A time interval associated with an event; its start and duration must follow an explicit protocol. |
| Mu rhythm | Sensorimotor oscillatory activity often studied around 8–13 Hz; the precise boundaries are analysis-dependent. |
| Beta rhythm | Sensorimotor activity often studied around 13–30 Hz; ranges vary between studies. |
| ERD/ERS | Relative decreases/increases in oscillatory power, requiring an appropriate reference interval and analysis design. |

**Scientific caution:** Scalp EEG activity and event labels do not themselves establish successful decoding, causal neural mechanisms or clinical efficacy. Later Notebooks will investigate filtering, ERD/ERS and classification under explicit evaluation protocols.

**Execution prerequisites:** Python with `numpy`, `pandas`, `scipy`, `matplotlib` and Jupyter. Open the Notebook with the repository checked out locally, or clone the repository in Colab and set the working directory inside that checkout before running cells. The original dataset files are read-only inputs and are never modified by this Notebook.

**01. Imports and Environment**

Import the scientific packages required for descriptive data inspection and confirm that the current working directory belongs to the repository. No GPU or model-training environment is needed.

In [ ]:
# ======================== 01. Imports and Environment ========================

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.io import loadmat

repo_candidates = (Path.cwd(), *Path.cwd().parents)
repository_root = next(
    (
        candidate
        for candidate in repo_candidates
        if (candidate / "Data").is_dir()
        and (candidate / "eeg_foundations_and_preprocessing").is_dir()
    ),
    None,
)
if repository_root is None:
    raise FileNotFoundError(
        "Repository root not found. Open the Notebook inside the cloned "
        "Motor Imagery repository, or change the working directory to it."
    )

print(f"Repository: {repository_root.name}")
print(f"NumPy: {np.__version__}; pandas: {pd.__version__}")


**02. Locate the Existing Calibration Recordings**

The repository already contains calibration files under `Data/`. We select the first matching filename in **sorted order** for a reproducible initial inspection. This does not mean that all recordings share the same subject identity, imagined-movement classes or recording quality. Later Notebook stages may inspect all suitable recordings separately.

In [ ]:
# ======================= 02. Locate Calibration Files ========================

data_directory = repository_root / "Data"
calibration_files = sorted(data_directory.glob("BCICIV_calib_ds1*_100Hz.mat"))
if not calibration_files:
    raise FileNotFoundError(
        f"No BCI Competition IV Dataset 1 calibration files found in {data_directory}."
    )

dataset_file = calibration_files[0]
display(pd.DataFrame({"Available calibration file": [p.name for p in calibration_files]}))
print(f"Selected for exploratory inspection: {dataset_file.name}")


**03. Inspect the MATLAB File Structure**

The published Dataset 1 file description documents `cnt` (continuous samples × channels), `mrk` (cue markers) and `nfo` (recording metadata) for calibration recordings. Here we check the **actual keys** in the selected file, rather than assuming that every `.mat` file has the same contents. The `simplify_cells=True` option converts common MATLAB cell/struct nesting to Python-friendly objects.

In [ ]:
# ========================= 03. Inspect MATLAB Source =========================

mat_data = loadmat(dataset_file, simplify_cells=True)
source_keys = sorted(key for key in mat_data if not key.startswith("__"))
required_keys = {"cnt", "mrk", "nfo"}
missing_keys = required_keys.difference(source_keys)
if missing_keys:
    raise KeyError(f"Required MATLAB variables are missing: {sorted(missing_keys)}")

for key in sorted(required_keys):
    value = mat_data[key]
    shape = getattr(value, "shape", None)
    print(f"{key}: type={type(value).__name__}, shape={shape}")

if not isinstance(mat_data["nfo"], dict) or not isinstance(mat_data["mrk"], dict):
    raise TypeError("Expected `nfo` and `mrk` to be MATLAB structures.")

print(f"Metadata fields: {sorted(mat_data['nfo'])}")
print(f"Marker fields: {sorted(mat_data['mrk'])}")


**04. Read EEG Signals and Recording Metadata**

The Dataset 1 description states that `cnt` is stored as signed integers and converted to microvolts using `0.1 × cnt`. The code checks the source type before this conversion to avoid silently applying the same multiplier to unrelated data. Channel labels and class names are taken from `nfo`, and the sampling frequency is read from the file instead of being hard-coded.

In [ ]:
# ====================== 04. Extract Recording Metadata =======================

raw_counts = np.asarray(mat_data["cnt"])
if raw_counts.ndim != 2 or raw_counts.dtype.kind not in "iu":
    raise ValueError(
        "The selected `cnt` must be a two-dimensional integer EEG array; "
        "verify the source format and scaling before continuing."
    )

eeg_uv = raw_counts.astype(np.float64) * 0.1
metadata = mat_data["nfo"]
required_metadata = {"fs", "clab", "classes"}
missing_metadata = required_metadata.difference(metadata)
if missing_metadata:
    raise KeyError(f"Missing recording metadata: {sorted(missing_metadata)}")

fs_value = float(np.asarray(metadata["fs"]).squeeze())
if not np.isfinite(fs_value) or fs_value <= 0:
    raise ValueError("Sampling frequency must be positive and finite.")


def flatten_labels(items):
    """Flatten MATLAB text entries into readable Python strings."""
    result = []
    for item in np.asarray(items, dtype=object).ravel():
        if isinstance(item, np.ndarray):
            result.extend(flatten_labels(item))
        elif isinstance(item, bytes):
            result.append(item.decode("utf-8").strip())
        else:
            result.append(str(item).strip())
    return result


channel_names = flatten_labels(metadata["clab"])
class_names = flatten_labels(metadata["classes"])
number_of_samples, number_of_channels = eeg_uv.shape
recording_duration_s = number_of_samples / fs_value

recording_summary = pd.DataFrame(
    {
        "Property": [
            "Selected source", "Samples", "EEG channels", "Sampling rate (Hz)",
            "Recording duration (s)", "Source data type", "Signal unit",
        ],
        "Value": [
            dataset_file.name, number_of_samples, number_of_channels,
            fs_value, round(recording_duration_s, 2),
            str(raw_counts.dtype), "µV (0.1 × stored counts)",
        ],
    }
)
display(recording_summary)
print(f"First ten channel labels: {channel_names[:10]}")
print(f"Class names in metadata: {class_names}")


**05. Interpret Cue Events and Class Metadata**

`mrk.pos` contains cue locations as **MATLAB sample positions** (1-based). Python arrays use zero-based indices, so subtract one before referencing the EEG array. `mrk.y` uses dataset-specific numerical class codes; for this published calibration format, `-1` identifies the first class in `nfo.classes`, and `+1` the second. The **movement names must be read from the selected file**; not every recording is necessarily left-hand versus right-hand imagery.

A cue time is not automatically a validated epoch onset or a physical movement measurement. Trial extraction and baseline selection are outside this Notebook.

In [ ]:
# ========================== 05. Inspect Cue Events ===========================

markers = mat_data["mrk"]
if "pos" not in markers or "y" not in markers:
    raise KeyError("The calibration file must provide `mrk.pos` and `mrk.y`.")

raw_positions = np.asarray(markers["pos"]).reshape(-1)
raw_labels = np.asarray(markers["y"]).reshape(-1)
if raw_positions.size != raw_labels.size or raw_positions.size == 0:
    raise ValueError("Event position and class arrays must be non-empty and aligned.")
if not np.isfinite(raw_positions).all() or not np.all(raw_positions == np.floor(raw_positions)):
    raise ValueError("MATLAB event positions must contain finite integer sample values.")

event_indices = raw_positions.astype(np.int64) - 1
class_codes = raw_labels.astype(np.int64)
event_seconds = event_indices / fs_value
class_lookup = {-1: class_names[0], 1: class_names[1]} if len(class_names) == 2 else {}

event_table = pd.DataFrame(
    {
        "MATLAB sample": raw_positions.astype(np.int64),
        "Python sample index": event_indices,
        "Time (s)": event_seconds,
        "Class code": class_codes,
        "Class name": [class_lookup.get(int(v), "Unresolved") for v in class_codes],
    }
)
display(event_table.head(10))
display(event_table.groupby(["Class code", "Class name"], dropna=False).size().rename("Cue count").reset_index())


**06. Visualise Example EEG Channels**

Plot a short continuous segment in **microvolts**. Prefer conventional sensorimotor labels `C3`, `Cz` and `C4` when these labels are present; otherwise show the first three available channels. These plots are descriptive. Without an event-aligned and properly referenced analysis they do not establish the presence or absence of ERD/ERS.

In [ ]:
# ======================= 06. Visualise Continuous EEG ========================

preferred_channels = [name for name in ("C3", "Cz", "C4") if name in channel_names]
selected_channels = preferred_channels if preferred_channels else channel_names[:3]
channel_indices = [channel_names.index(name) for name in selected_channels]
plot_samples = min(number_of_samples, int(round(5 * fs_value)))
plot_times = np.arange(plot_samples) / fs_value

fig, axes = plt.subplots(len(channel_indices), 1, figsize=(11, 6), sharex=True)
for ax, name, index in zip(np.atleast_1d(axes), selected_channels, channel_indices):
    ax.plot(plot_times, eeg_uv[:plot_samples, index], linewidth=0.8)
    ax.set_ylabel(f"{name} (µV)")
    ax.grid(alpha=0.2)
axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"Exploratory EEG segment: {dataset_file.name}")
fig.tight_layout()
plt.show()


**07. Visualise the Cue Timeline**

Plot the cue times by class code to inspect the timing and broad distribution of recorded prompts. The markers describe the experiment annotations, not the success of a participant's imagery or the output of a decoding model.

In [ ]:
# ======================== 07. Visualise Cue Timeline =========================

fig, ax = plt.subplots(figsize=(11, 3.8))
for code_value in sorted(np.unique(class_codes)):
    mask = class_codes == code_value
    label = f"{code_value}: {class_lookup.get(int(code_value), 'Unresolved')}"
    ax.scatter(event_seconds[mask], class_codes[mask], s=13, alpha=0.75, label=label)
ax.set(xlabel="Time (s)", ylabel="Recorded class code", title="Motor imagery cue timeline")
ax.set_yticks(sorted(np.unique(class_codes)))
ax.grid(axis="x", alpha=0.25)
ax.legend(title="Cue annotation", loc="best")
fig.tight_layout()
plt.show()


**08. Scientific Quality Gate**

The following checks must pass before treating this Notebook's source inspection as technically complete. They verify data shape, scaling prerequisites, channel metadata, class labels and event indexing. They do **not** validate the participant's performance, preprocessing quality, classifier accuracy or cross-subject generalisation.

In [ ]:
# ======================== 08. Scientific Quality Gate ========================

quality_checks = {
    "EEG has samples and channels": eeg_uv.ndim == 2 and min(eeg_uv.shape) > 0,
    "Channel labels match EEG channels": len(channel_names) == number_of_channels,
    "EEG values are finite": bool(np.isfinite(eeg_uv).all()),
    "Valid sampling frequency": bool(np.isfinite(fs_value) and fs_value > 0),
    "Two named imagery classes": len(class_names) == 2 and all(class_names),
    "Cue positions and labels align": len(event_indices) == len(class_codes) > 0,
    "MATLAB cue indices are in range": bool(np.all(event_indices >= 0) and np.all(event_indices < number_of_samples)),
    "Cue codes follow documented values": set(np.unique(class_codes)).issubset({-1, 1}),
    "Cue positions are chronological": bool(np.all(np.diff(event_indices) >= 0)),
}
quality_table = pd.DataFrame(
    {"Scientific/technical check": list(quality_checks), "Pass": list(quality_checks.values())}
)
display(quality_table)
failed_checks = [name for name, passed in quality_checks.items() if not passed]
if failed_checks:
    raise AssertionError(f"Scientific Quality Gate failed: {failed_checks}")
print("Quality Gate passed for the selected source recording only.")


**Interpretation and Next Stage**

After running this Notebook on the original recording, interpret the displayed metadata, signal characteristics and cue counts using the **actual outputs**. If any source check fails, investigate the file schema or assumptions instead of forcing a passing result. Avoid generalising from one recording to the full dataset.

The next Notebook can investigate dataset-level comparisons, channel locations and event-aligned trial preparation once this initial source inspection has been executed and approved. Temporal/spatial filtering, CSP and supervised decoding remain separate research stages. A final, stage-specific SVG workflow will be prepared after scientific approval of this Notebook.